# Technology Skills Consolidation

This notebook reads `Technology Skills.xlsx`, keeps only `O*NET-SOC Code`, `Title`, and `Example`, and aggregates top examples per O*NET/base SOC code.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
tech_path = ROOT / 'reference_data' / 'Technology Skills.xlsx'
out_path = ROOT.parent / 'site' / 'public' / 'soc_tech_skills.json'

df = pd.read_excel(tech_path, usecols=['O*NET-SOC Code', 'Title', 'Example'])
df = df.rename(columns={'O*NET-SOC Code': 'onet_soc_code', 'Title': 'title', 'Example': 'example'})
df['onet_soc_code'] = df['onet_soc_code'].astype(str).str.strip()
df['base_soc'] = df['onet_soc_code'].str.split('.').str[0]
df['example'] = df['example'].astype(str).str.strip()
df = df[(df['onet_soc_code'] != '') & (df['example'] != '')]

print('Rows:', len(df))
print('O*NET codes:', df['onet_soc_code'].nunique())
print('Base SOC codes:', df['base_soc'].nunique())

In [ ]:
by_onet = (
    df.groupby(['onet_soc_code', 'example'], as_index=False).size()
    .sort_values(['onet_soc_code', 'size'], ascending=[True, False])
)

by_base = (
    df.groupby(['base_soc', 'example'], as_index=False).size()
    .sort_values(['base_soc', 'size'], ascending=[True, False])
)

payload = {
    'by_onet': {
        code: group['example'].head(25).tolist()
        for code, group in by_onet.groupby('onet_soc_code')
    },
    'by_base_soc': {
        code: group['example'].head(30).tolist()
        for code, group in by_base.groupby('base_soc')
    },
}

out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Wrote:', out_path)
print('by_onet:', len(payload['by_onet']))
print('by_base_soc:', len(payload['by_base_soc']))